In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("IntermediateRDD").getOrCreate()
sc = spark.sparkContext


# 1️⃣ Word Count from Text File (flatMap + reduceByKey)

In [4]:
rdd = sc.textFile("words.txt")

word_counts = (
    rdd.flatMap(lambda line: line.split())
       .map(lambda word: (word, 1))
       .reduceByKey(lambda a, b: a + b)
)

print(word_counts.collect())


[('"file":', 1), ('"words.txt",', 1), ('"content":', 1), ('"hello', 1), ('world\\nhello', 1), ('spark\\nhello', 1), ('{', 1), ('python"', 1), ('}', 1)]


# 2️⃣ Log File Processing (filter + map)

In [5]:
rdd = sc.textFile("logs.txt")

errors = (
    rdd.filter(lambda line: "ERROR" in line)
       .map(lambda line: line.replace("ERROR ", ""))
)

print(errors.collect())


['  "content": "INFO Start\\nDisk failure\\nWARN Low memory\\nNetwork timeout"']


# 3️⃣ Compute Average Numbers from Text File (map + reduce)

In [9]:
rdd = sc.textFile("numbers.txt")

# Clean and convert safely
rdd_numbers = (
    rdd.map(lambda x: x.strip())          # remove spaces
       .filter(lambda x: x != "")         # remove blank lines
       .map(lambda x: int(x))             # convert to int
)

total = rdd_numbers.reduce(lambda a, b: a + b)
count = rdd_numbers.count()
average = total / count

print("Average:", average)


Average: 30.0


# 4️⃣ Category Count (countByValue)

In [10]:
rdd = sc.textFile("categories.txt")

category_counts = rdd.countByValue()

print(category_counts)


defaultdict(<class 'int'>, {'IT': 3, 'HR': 1, 'Finance': 2})


# 5️⃣ Manual CSV Parsing (map + filter)

In [11]:
rdd = sc.textFile("employees.csv")

parsed = (
    rdd.map(lambda line: line.split(","))
       .map(lambda fields: (fields[0], fields[1], int(fields[2])))
)

high_salary = parsed.filter(lambda x: x[2] > 80000)

print(high_salary.collect())


[('John', 'IT', 90000)]
